# Model 7: EfficientNet-B3 — Siamese Multimodal Network

**Kiến trúc:**
```
EfficientNet-B3 (shared, pretrained ImageNet) → GAP
    → Linear(feat_dim, 128) + ReLU + Dropout(0.5)   # per eye

[left_128D | right_128D] → 256-D image feature
[age_norm, gender_M, gender_F] → Linear(3, 16) + ReLU → 16-D

[256-D | 16-D] = 272-D
    → Linear(272, 64) + BatchNorm1d(64) + ReLU + Dropout(0.5)
    → Linear(64, 8)   ← logits (SmoothBCEWithLogitsLoss)
```

**Training strategy (2 phase):**
- Phase 1 (epoch 1–5): backbone frozen, chỉ train head với LR=1e-3
- Phase 2 (epoch 6+): unfreeze toàn bộ với differential LR — backbone 3e-5, head 3e-4

**Loss**: SmoothBCEWithLogitsLoss (label smoothing=0.1) với pos_weight (class imbalance)  
**Optimizer**: Adam + ReduceLROnPlateau  
**Regularization**: Dropout(0.5), WeightDecay=5e-4, GradientClip=1.0, BatchNorm

In [1]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# import torch
# print("PyTorch version:", torch.__version__)
# print("CUDA version (built with):", torch.version.cuda)
# print("GPU:", torch.cuda.get_device_name(0))
# print("Compute capability:", torch.cuda.get_device_capability(0))

In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cpu


In [3]:
# ===== CẤU HÌNH =====
PREPROC_DIR  = f'/kaggle/input/datasets/xuanductran/odir-data-preprocess-augmentation/odir-data-preprocess-augmentation'
MODEL_DIR    = f'/kaggle/working'
CHECKPT_DIR  = f'{MODEL_DIR}/checkpoints'
HISTORY_DIR  = f'{MODEL_DIR}/history'
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

TARGET_COLS   = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
BACKBONE      = 'efficientnet_b3'
BACKBONE_TAG  = 'efficientnetb3'   # dùng đặt tên file
IMAGE_SIZE    = 300
BATCH_SIZE    = 16
NUM_EPOCHS    = 50
FREEZE_EPOCHS = 5       # số epoch freeze backbone (phase 1)
LR_HEAD       = 1e-3    # LR khi chỉ train head (backbone frozen)
LR_FINETUNE   = 3e-5    # LR khi unfreeze toàn bộ (backbone)
WEIGHT_DECAY  = 5e-4
DROPOUT       = 0.5
PROJ_DIM      = 128
TABULAR_DIM   = 16
NUM_WORKERS   = 4
PATIENCE      = 10      # early stopping patience
GRAD_CLIP     = 1.0     # gradient clipping max norm

print(f'Backbone: {BACKBONE}, Image size: {IMAGE_SIZE}x{IMAGE_SIZE}, Batch: {BATCH_SIZE}')
print(f'Training: {FREEZE_EPOCHS} freeze epochs → unfreeze with LR={LR_FINETUNE}')

Backbone: efficientnet_b3, Image size: 300x300, Batch: 16
Training: 5 freeze epochs → unfreeze with LR=3e-05


## 1. Load dữ liệu

In [4]:
meta_df = pd.read_csv(f'{PREPROC_DIR}/split_metadata.csv')
OLD_PREFIX = '/home/centrala/work/ou/kltn/Ocular-Disease-Recognition/odir-data-preprocess-augmentation'

meta_df['left_path']  = meta_df['left_path'].str.replace(OLD_PREFIX, PREPROC_DIR, regex=False)
meta_df['right_path'] = meta_df['right_path'].str.replace(OLD_PREFIX, PREPROC_DIR, regex=False)
with open(f'{PREPROC_DIR}/class_weights.json') as f:
    class_weights_dict = json.load(f)

pos_weight = torch.tensor(
    [class_weights_dict[c] for c in TARGET_COLS],
    dtype=torch.float32
).to(device)

train_df = meta_df[meta_df['split'] == 'train'].reset_index(drop=True)
val_df   = meta_df[(meta_df['split'] == 'val') & (meta_df['aug_id'] == -1)].reset_index(drop=True)

print(f'Train rows (incl. augmented): {len(train_df)}')
print(f'Val rows (base only):         {len(val_df)}')
print(f'Class weights: {class_weights_dict}')

Train rows (incl. augmented): 17171
Val rows (base only):         697
Class weights: {'N': 2.0739, 'D': 2.109, 'G': 15.245, 'C': 15.4631, 'A': 20.3304, 'H': 33.0694, 'M': 19.1066, 'O': 2.581}


In [5]:
# Chuẩn hóa tuổi: Z-score từ base train
train_base_df = meta_df[(meta_df['split'] == 'train') & (meta_df['aug_id'] == -1)]
AGE_MEAN = float(train_base_df['age'].mean())
AGE_STD  = float(train_base_df['age'].std())

norm_params = {'age_mean': AGE_MEAN, 'age_std': AGE_STD}
with open(f'{MODEL_DIR}/norm_params.json', 'w') as f:
    json.dump(norm_params, f, indent=2)
print(f'Age normalization — mean: {AGE_MEAN:.2f}, std: {AGE_STD:.2f}')
print(f'Saved: {MODEL_DIR}/norm_params.json')


def encode_gender(gender_str):
    """[1, 0] = Male, [0, 1] = Female"""
    s = str(gender_str).strip().lower()
    if s in ('male', 'm'):
        return [1.0, 0.0]
    return [0.0, 1.0]

Age normalization — mean: 57.82, std: 11.77
Saved: /kaggle/working/norm_params.json


## 2. Dataset và DataLoader

In [6]:
class OdirDataset(Dataset):
    """
    Load ảnh đã tiền xử lý (Graham + resize) từ đĩa.
    Chỉ normalize theo ImageNet stats ở đây.
    """
    def __init__(self, df, target_cols, image_size, age_mean, age_std):
        self.df          = df.reset_index(drop=True)
        self.target_cols = target_cols
        self.age_mean    = age_mean
        self.age_std     = age_std
        self.transform   = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        left_img  = Image.open(row['left_path']).convert('RGB')
        right_img = Image.open(row['right_path']).convert('RGB')
        left_t    = self.transform(left_img)
        right_t   = self.transform(right_img)

        age_norm  = (row['age'] - self.age_mean) / (self.age_std + 1e-8)
        gender    = encode_gender(row['gender'])
        tabular   = torch.tensor([age_norm] + gender, dtype=torch.float32)

        labels    = torch.tensor([float(row[c]) for c in self.target_cols],
                                  dtype=torch.float32)
        return left_t, right_t, tabular, labels


train_dataset = OdirDataset(train_df, TARGET_COLS, IMAGE_SIZE, AGE_MEAN, AGE_STD)
val_dataset   = OdirDataset(val_df,   TARGET_COLS, IMAGE_SIZE, AGE_MEAN, AGE_STD)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 1074 | Val batches: 44


## 3. Kiến trúc mô hình

In [7]:
class SiameseMultimodalNet(nn.Module):
    """
    Multimodal Dual-Eye Siamese Network.

    Backbone (shared weights) xử lý ảnh mắt trái và mắt phải độc lập,
    sau đó kết hợp với đặc trưng nhân khẩu học (tuổi + giới tính).

    Output: logits (8 chiều). Áp dụng sigmoid() khi inference.
    Dùng SmoothBCEWithLogitsLoss khi training.
    """
    def __init__(self, backbone_name, proj_dim=128, tabular_dim=16,
                 dropout=0.4, num_classes=8):
        super().__init__()
        self.backbone  = timm.create_model(backbone_name, pretrained=True,
                                           num_classes=0, global_pool='avg')
        feat_dim       = self.backbone.num_features

        self.projector = nn.Sequential(
            nn.Linear(feat_dim, proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        self.tabular_encoder = nn.Sequential(
            nn.Linear(3, tabular_dim),
            nn.ReLU(inplace=True)
        )
        fused_dim    = 2 * proj_dim + tabular_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward_one(self, x):
        return self.projector(self.backbone(x))

    def forward(self, left, right, tabular):
        img_feat = torch.cat([self.forward_one(left),
                               self.forward_one(right)], dim=1)
        tab_feat = self.tabular_encoder(tabular)
        fused    = torch.cat([img_feat, tab_feat], dim=1)
        return self.classifier(fused)


# Kiểm tra backbone khả dụng
available = timm.list_models(f'*{BACKBONE_TAG}*', pretrained=True)
print(f'Available {BACKBONE_TAG} models: {available[:5]}')

model = SiameseMultimodalNet(
    backbone_name=BACKBONE,
    proj_dim=PROJ_DIM,
    tabular_dim=TABULAR_DIM,
    dropout=DROPOUT,
    num_classes=len(TARGET_COLS)
).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

Available efficientnetb3 models: []


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Total parameters:     10,911,152
Trainable parameters: 10,911,152


## 4. Loss, Optimizer, Scheduler

In [8]:
class SmoothBCEWithLogitsLoss(nn.Module):
    """BCEWithLogitsLoss với label smoothing để giảm overconfidence."""
    def __init__(self, pos_weight=None, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets)


criterion = SmoothBCEWithLogitsLoss(pos_weight=pos_weight, smoothing=0.1)

# Phase 1: freeze backbone, chỉ train head
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4
)

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Loss: SmoothBCEWithLogitsLoss (smoothing=0.1) với pos_weight')
print(f'Phase 1: backbone frozen | head LR={LR_HEAD} | wd={WEIGHT_DECAY}')
print(f'Trainable params (head only): {trainable_now:,}')

Loss: SmoothBCEWithLogitsLoss (smoothing=0.1) với pos_weight
Phase 1: backbone frozen | head LR=0.001 | wd=0.0005
Trainable params (head only): 214,920


## 5. Hàm Train và Validate

In [9]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_logits, all_labels = [], []

    for left, right, tabular, labels in loader:
        left, right, tabular, labels = (
            left.to(device), right.to(device),
            tabular.to(device), labels.to(device)
        )
        optimizer.zero_grad()
        logits = model(left, right, tabular)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.cpu())

    avg_loss   = total_loss / len(loader)
    all_probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    all_labels = torch.cat(all_labels).numpy()
    try:
        auc = roc_auc_score(all_labels, all_probs, average='macro')
    except Exception:
        auc = 0.0
    return avg_loss, auc


def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    with torch.no_grad():
        for left, right, tabular, labels in loader:
            left, right, tabular, labels = (
                left.to(device), right.to(device),
                tabular.to(device), labels.to(device)
            )
            logits = model(left, right, tabular)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())

    avg_loss   = total_loss / len(loader)
    all_probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    all_labels = torch.cat(all_labels).numpy()
    try:
        auc = roc_auc_score(all_labels, all_probs, average='macro')
    except Exception:
        auc = 0.0
    return avg_loss, auc


print('Hàm training và validation OK (gradient clipping enabled)')

Hàm training và validation OK (gradient clipping enabled)


## 6. Training Loop

In [10]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_auc': [], 'val_auc': [],
    'lr': []
}
best_val_loss    = float('inf')
patience_counter = 0
checkpoint_path  = f'{CHECKPT_DIR}/{BACKBONE_TAG}_best.pth'

print(f'Bắt đầu training: {NUM_EPOCHS} epochs ({FREEZE_EPOCHS} freeze + {NUM_EPOCHS - FREEZE_EPOCHS} finetune)')
print(f'Checkpoint: {checkpoint_path}')
print('-' * 90)

for epoch in range(1, NUM_EPOCHS + 1):

    # === Phase 2: unfreeze backbone sau FREEZE_EPOCHS ===
    if epoch == FREEZE_EPOCHS + 1:
        print(f'\n[Epoch {epoch}] Unfreezing backbone — switching to differential LR')
        for param in model.backbone.parameters():
            param.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.backbone.parameters(),        'lr': LR_FINETUNE},
            {'params': model.projector.parameters(),       'lr': LR_FINETUNE * 10},
            {'params': model.tabular_encoder.parameters(), 'lr': LR_FINETUNE * 10},
            {'params': model.classifier.parameters(),      'lr': LR_FINETUNE * 10},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=4
        )
        print(f'  backbone LR={LR_FINETUNE}, head LR={LR_FINETUNE * 10}')
        print('-' * 90)

    t0 = time.time()
    train_loss, train_auc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_auc   = validate_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    history['lr'].append(current_lr)

    elapsed = time.time() - t0
    phase   = 'FREEZE' if epoch <= FREEZE_EPOCHS else 'FINETUNE'
    print(f'[{phase}] Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train Loss={train_loss:.4f} AUC={train_auc:.4f} | '
          f'Val Loss={val_loss:.4f} AUC={val_auc:.4f} | '
          f'LR={current_lr:.2e} | {elapsed:.0f}s')

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save({
            'epoch':                 epoch,
            'model_state_dict':      model.state_dict(),
            'optimizer_state_dict':  optimizer.state_dict(),
            'val_loss':              val_loss,
            'val_auc':               val_auc,
            'backbone':              BACKBONE,
            'config': {
                'image_size': IMAGE_SIZE, 'proj_dim': PROJ_DIM,
                'tabular_dim': TABULAR_DIM, 'dropout': DROPOUT
            }
        }, checkpoint_path)
        print(f'  *** Best checkpoint saved (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping sau {epoch} epochs (patience={PATIENCE})')
            break

print(f'\nBest val_loss: {best_val_loss:.4f}')

Bắt đầu training: 50 epochs (5 freeze + 45 finetune)
Checkpoint: /kaggle/working/checkpoints/efficientnetb3_best.pth
------------------------------------------------------------------------------------------


[FREEZE] Epoch   1/50 | Train Loss=1.3787 AUC=0.7222 | Val Loss=1.3359 AUC=0.7743 | LR=1.00e-03 | 6186s
  *** Best checkpoint saved (val_loss=1.3359)


[FREEZE] Epoch   2/50 | Train Loss=1.3294 AUC=0.7867 | Val Loss=1.4052 AUC=0.7902 | LR=1.00e-03 | 5562s


[FREEZE] Epoch   3/50 | Train Loss=1.3172 AUC=0.8024 | Val Loss=1.3396 AUC=0.7935 | LR=1.00e-03 | 5532s


[FREEZE] Epoch   4/50 | Train Loss=1.3089 AUC=0.8136 | Val Loss=1.3092 AUC=0.7961 | LR=1.00e-03 | 5446s
  *** Best checkpoint saved (val_loss=1.3092)


[FREEZE] Epoch   5/50 | Train Loss=1.3024 AUC=0.8185 | Val Loss=1.3501 AUC=0.8043 | LR=1.00e-03 | 5359s

[Epoch 6] Unfreezing backbone — switching to differential LR
  backbone LR=3e-05, head LR=0.00030000000000000003
------------------------------------------------------------------------------------------


[FINETUNE] Epoch   6/50 | Train Loss=1.2664 AUC=0.8571 | Val Loss=1.2866 AUC=0.8310 | LR=3.00e-05 | 9874s


## 7. Lưu lịch sử và vẽ đồ thị

In [ ]:
history_path = f'{HISTORY_DIR}/{BACKBONE_TAG}_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Saved: {history_path}')

epochs = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], label='Train Loss', color='#4C72B0')
ax1.plot(epochs, history['val_loss'],   label='Val Loss',   color='#DD8452')
ax1.axvline(x=FREEZE_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
ax1.set_title(f'Loss — {BACKBONE}', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_auc'], label='Train Macro AUC', color='#4C72B0')
ax2.plot(epochs, history['val_auc'],   label='Val Macro AUC',   color='#DD8452')
ax2.axvline(x=FREEZE_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
ax2.set_title(f'AUC-ROC — {BACKBONE}', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro AUC')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
chart_path = f'{MODEL_DIR}/chart_{BACKBONE_TAG}_history.png'
plt.savefig(chart_path, dpi=120)
plt.show()
print(f'Saved: {chart_path}')

## Tóm tắt

**Backbone**: EfficientNet-B3 (`efficientnet_b3`) — ImageNet pretrained, ~12M params  
**Image size**: 300×300

**Cải tiến (so với baseline):**
- 2-phase training: freeze backbone 5 epoch → unfreeze với differential LR
- Label smoothing (smoothing=0.1) giảm overconfidence
- BatchNorm1d trong classifier ổn định training
- Dropout=0.5, WeightDecay=5e-4
- Gradient clipping (max_norm=1.0)
- NUM_EPOCHS=50, PATIENCE=10

**Outputs:**
- `checkpoints/efficientnetb3_best.pth`
- `history/efficientnetb3_history.json`
- `norm_params.json`
- `chart_efficientnetb3_history.png`

**Bước tiếp theo:** Chạy `07_evaluation.ipynb` để so sánh tất cả các mô hình